# Initial Questions

1. Our corpus will have 12 documents
2. The average document length is
3. The min length is: and the max length is:
4. The OHCO structure varies based on the novel in the corpus. For The Mysteries of Udolpho, it is ['vol_num', 'chap_num', 'para_num', 'sent_num', 'token_num'] ,for THe Old ENglish Baron, it is ['para_num', 'sent_num', 'token_num'], for the Castle of Otranto, it is ['chap_num', 'para_num', 'sent_num', 'token_num'], for Melmoth the Wanderer, it is ['chap_num', 'para_num', 'sent_num', 'token_num']

In [24]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='ticks')

In [25]:
data_home = '../input/datasets/mohiniguptarde/'
output_dir = '../working'

In [26]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

# High Gothic

## The Mysteries of Udolpho

token/vocab

In [27]:
src_file = f"{data_home}/high-gothic/pg3268.txt"


In [28]:
OHCO = ['vol_num', 'chap_num', 'para_num', 'sent_num', 'token_num']

In [29]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 95
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]
vol_pat = r"^\s*(?:volume)\s+\d+"
chap_pat = r"^\s*(?:chapter)\s+[IVXLCDM]+"
vol_lines = LINES.line_str.str.match(vol_pat, case=False) # Returns a truth vector
#LINES.loc[vol_lines] # Use as filter for dataframe
LINES.loc[vol_lines, 'vol_num'] = [i+1 for i in range(LINES.loc[vol_lines].shape[0])]
LINES.vol_num = LINES.vol_num.ffill()

In [30]:
LINES = LINES.dropna(subset=['vol_num']) # Remove everything before Chapter 1
LINES = LINES.loc[~vol_lines] # Remove chapter heading lines; their work is done
LINES.vol_num = LINES.vol_num.astype('int') # Convert chap_num from float to int

In [31]:
VOLS = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('vol_str')
VOLS['vol_str'] = VOLS.vol_str.str.strip()


In [32]:
#VOLS

In [33]:
chap_lines = LINES.line_str.str.match(chap_pat, case=False) # Returns a truth vector
LINES.loc[chap_lines, 'chap_num'] = (LINES.loc[chap_lines].groupby('vol_num').cumcount() + 1)
LINES['chap_num'] = LINES['chap_num'].ffill()
LINES = LINES.dropna(subset=['chap_num'])
LINES = LINES.loc[~chap_lines]
LINES['chap_num'] = LINES['chap_num'].astype(int)
#LINES.sample(5)

In [34]:
CHAPS = LINES.groupby(OHCO[:2])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('chap_str')
CHAPS['chap_str'] = CHAPS.chap_str.str.strip()


In [35]:
para_pat = r'\n\n+'
PARAS = CHAPS['chap_str'].str.split(para_pat, expand=True).stack()\
    .to_frame('para_str').sort_index()
PARAS.index.names = OHCO[:3]

In [36]:
#PARAS.head()

In [37]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [38]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:4]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [39]:
keep_whitespace = True

if keep_whitespace:
    # Return a tokenized copy of text
    # using NLTK's recommended word tokenizer.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')
else:
    # Tokenize a string on whitespace (space, tab, newline).
    # In general, users should use the string ``split()`` method instead.
    # Returns fewer tokens.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')

TOKENS.index.names = OHCO[:5]

TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"\W+", "", regex=True)
TOKENS

TOKENS['pos_group'] = TOKENS.pos.str[:2]
TOKENS.head()

pos_tuple  pos token_str  \
vol_num chap_num para_num sent_num token_num                                
1       1        0        0        0            (home, NN)   NN      home   
                                   1             (is, VBZ)  VBZ        is   
                                   2             (the, DT)   DT       the   
                                   3          (resort, NN)   NN    resort   
                                   4              (Of, IN)   IN        Of   

                                             term_str pos_group  
vol_num chap_num para_num sent_num token_num                     
1       1        0        0        0             home        NN  
                                   1               is        VB  
                                   2              the        DT  
                                   3           resort        NN  
                                   4               of        IN

In [40]:
# token_pat = r"[\s',-]+"
# TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
#     .to_frame('token_str')
# TOKENS.index.names = OHCO[:5]
# TOKENS
# TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
# VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
# VOCAB.index.name = 'term_id'
# VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [41]:
TOKENS = TOKENS[TOKENS.index.get_level_values('vol_num') == 1]
TOKENS = TOKENS.droplevel('vol_num')

In [42]:
udolpho_VOLS = VOLS
udolpho_CHAPS = CHAPS
udolpho_PARAS = PARAS
udolpho_SENTS = SENTS
udolpho_TOKENS = TOKENS
#udolpho_VOCAB = VOCAB

In [43]:
udolpho_TOKENS.to_csv(f"{output_dir}/udolpho-TOKENS.csv")


## The Old English Baron: a Gothic Story

In [44]:
src_file = f"{data_home}/high-gothic/pg5182.txt"


In [45]:
OHCO = ['para_num', 'sent_num', 'token_num']

In [46]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]

In [47]:
para_pat = r'\n\n+'
PARAS = LINES['line_str'].str.split(para_pat, expand=True).stack().reset_index(drop=True)\
    .to_frame('para_str')
PARAS.index.names = OHCO[:1]

In [48]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [49]:
PARAS.head()

,para_str
para_num,
4,Produced by Jack Voller
13,THE OLD ENGLISH BARON
16,By Clara Reeve
21,PREFACE
25,"As this Story is of a species which, though no..."


In [50]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:2]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [51]:
keep_whitespace = True

if keep_whitespace:
    # Return a tokenized copy of text
    # using NLTK's recommended word tokenizer.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')
else:
    # Tokenize a string on whitespace (space, tab, newline).
    # In general, users should use the string ``split()`` method instead.
    # Returns fewer tokens.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')

TOKENS.index.names = OHCO[:5]

TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"\W+", "", regex=True)
TOKENS

TOKENS['pos_group'] = TOKENS.pos.str[:2]
TOKENS.head()

pos_tuple  pos token_str  term_str  \
para_num sent_num token_num                                             
4        0        0          (Produced, VBN)  VBN  Produced  produced   
                  1                 (by, IN)   IN        by        by   
                  2              (Jack, NNP)  NNP      Jack      jack   
                  3            (Voller, NNP)  NNP    Voller    voller   
13       0        0                (THE, DT)   DT       THE       the   

                            pos_group  
para_num sent_num token_num            
4        0        0                VB  
                  1                IN  
                  2                NN  
                  3                NN  
13       0        0                DT

In [52]:
# token_pat = r"[\s',-]+"
# TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
#     .to_frame('token_str')
# TOKENS.index.names = OHCO[:5]
# TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
# VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
# VOCAB.index.name = 'term_id'
# VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [53]:
TOKENS = pd.concat([TOKENS], keys=[1], names=['chap_num'])

In [54]:
TOKENS.head()

pos_tuple  pos token_str  \
chap_num para_num sent_num token_num                                   
1        4        0        0          (Produced, VBN)  VBN  Produced   
                           1                 (by, IN)   IN        by   
                           2              (Jack, NNP)  NNP      Jack   
                           3            (Voller, NNP)  NNP    Voller   
         13       0        0                (THE, DT)   DT       THE   

                                      term_str pos_group  
chap_num para_num sent_num token_num                      
1        4        0        0          produced        VB  
                           1                by        IN  
                           2              jack        NN  
                           3            voller        NN  
         13       0        0               the        DT

In [55]:
baron_PARAS = PARAS
baron_SENTS = SENTS
baron_TOKENS = TOKENS
#baron_VOCAB = VOCAB

In [56]:
baron_TOKENS.to_csv(f"{output_dir}/baron-TOKENS.csv")


## The Castle of Otranto

In [57]:
src_file = f"{data_home}/high-gothic/pg696.txt"
OHCO = ['chap_num', 'para_num', 'sent_num', 'token_num']

In [58]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]

chap_pat = r"^\s*(?:chapter)\s+[IVXLCDM]+"
chap_lines = LINES.line_str.str.match(chap_pat, case=False) # Returns a truth vector

LINES.loc[chap_lines, 'chap_num'] = [i+1 for i in range(LINES.loc[chap_lines].shape[0])]
LINES.chap_num = LINES.chap_num.ffill()

In [59]:
LINES = LINES.dropna(subset=['chap_num']) # Remove everything before Chapter 1
LINES = LINES.loc[~chap_lines] # Remove chapter heading lines; their work is done
LINES.chap_num = LINES.chap_num.astype('int') # Convert chap_num from float to int

In [60]:
CHAPS = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('chap_str')
CHAPS['chap_str'] = CHAPS.chap_str.str.strip()


In [61]:
CHAPS.head()

,chap_str
chap_num,
1,"Manfred, Prince of Otranto, had one son and on..."
2,"Matilda, who by Hippolita’s order had retired ..."
3,Manfred’s heart misgave him when he beheld the...
4,The sorrowful troop no sooner arrived at the c...
5,Every reflection which Manfred made on the Fri...


In [62]:
para_pat = r'\n\n+'
PARAS = CHAPS['chap_str'].str.split(para_pat, expand=True).stack()\
    .to_frame('para_str').sort_index()
PARAS.index.names = OHCO[:2]

In [63]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [64]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:3]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [65]:
keep_whitespace = True

if keep_whitespace:
    # Return a tokenized copy of text
    # using NLTK's recommended word tokenizer.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')
else:
    # Tokenize a string on whitespace (space, tab, newline).
    # In general, users should use the string ``split()`` method instead.
    # Returns fewer tokens.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')

TOKENS.index.names = OHCO[:5]

TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"\W+", "", regex=True)
TOKENS

TOKENS['pos_group'] = TOKENS.pos.str[:2]
TOKENS.head()

pos_tuple  pos token_str term_str  \
chap_num para_num sent_num token_num                                           
1        0        0        0          (Manfred, NNP)  NNP   Manfred  manfred   
                           1                  (,, ,)    ,         ,            
                           2           (Prince, NNP)  NNP    Prince   prince   
                           3                (of, IN)   IN        of       of   
                           4          (Otranto, NNP)  NNP   Otranto  otranto   

                                     pos_group  
chap_num para_num sent_num token_num            
1        0        0        0                NN  
                           1                 ,  
                           2                NN  
                           3                IN  
                           4                NN

In [66]:
# token_pat = r"[\s',-]+"
# TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
#     .to_frame('token_str')
# TOKENS.index.names = OHCO[:4]
# TOKENS
# TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
# VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
# VOCAB.index.name = 'term_id'
# VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [67]:
TOKENS.head()

pos_tuple  pos token_str term_str  \
chap_num para_num sent_num token_num                                           
1        0        0        0          (Manfred, NNP)  NNP   Manfred  manfred   
                           1                  (,, ,)    ,         ,            
                           2           (Prince, NNP)  NNP    Prince   prince   
                           3                (of, IN)   IN        of       of   
                           4          (Otranto, NNP)  NNP   Otranto  otranto   

                                     pos_group  
chap_num para_num sent_num token_num            
1        0        0        0                NN  
                           1                 ,  
                           2                NN  
                           3                IN  
                           4                NN

In [68]:
otranto_CHAPS = CHAPS
otranto_PARAS = PARAS
otranto_SENTS = SENTS
otranto_TOKENS = TOKENS
#otranto_VOCAB = VOCAB

In [69]:
otranto_TOKENS.to_csv(f"{output_dir}/otranto-TOKENS.csv")


## Document Lengths for High Gothic Novels

In [70]:
print("Udolpho len: ", len(udolpho_TOKENS))
print("BAron len: ", len(baron_TOKENS))
print("Otranto len: ", len(otranto_TOKENS))
high_gothic_lens = [len(udolpho_TOKENS), len(baron_TOKENS), len(otranto_TOKENS)]
print("Average High GOthic len: ", (sum(high_gothic_lens)/3))

Udolpho len:  80568
BAron len:  63628
Otranto len:  40216
Average High GOthic len:  61470.666666666664


# Classic/Romantic Gothic

## Melmoth the Wanderer

In [71]:
src_file = f"{data_home}/melmoth/pg53685.txt"
OHCO = ['chap_num', 'para_num', 'sent_num', 'token_num']

In [72]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]

chap_pat = r"^\s*(?:chapter)\s+[IVXLCDM]+"
chap_lines = LINES.line_str.str.match(chap_pat, case=False) # Returns a truth vector

LINES.loc[chap_lines, 'chap_num'] = [i+1 for i in range(LINES.loc[chap_lines].shape[0])]
LINES.chap_num = LINES.chap_num.ffill()

In [73]:
LINES = LINES.dropna(subset=['chap_num']) # Remove everything before Chapter 1
LINES = LINES.loc[~chap_lines] # Remove chapter heading lines; their work is done
LINES.chap_num = LINES.chap_num.astype('int') # Convert chap_num from float to int

In [74]:
CHAPS = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('chap_str')
CHAPS['chap_str'] = CHAPS.chap_str.str.strip()


In [75]:
para_pat = r'\n\n+'
PARAS = CHAPS['chap_str'].str.split(para_pat, expand=True).stack()\
    .to_frame('para_str').sort_index()
PARAS.index.names = OHCO[:2]

In [76]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [77]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:3]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [78]:
keep_whitespace = True

if keep_whitespace:
    # Return a tokenized copy of text
    # using NLTK's recommended word tokenizer.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')
else:
    # Tokenize a string on whitespace (space, tab, newline).
    # In general, users should use the string ``split()`` method instead.
    # Returns fewer tokens.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')

TOKENS.index.names = OHCO[:5]

TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"\W+", "", regex=True)
TOKENS

TOKENS['pos_group'] = TOKENS.pos.str[:2]
TOKENS.head()

pos_tuple  pos token_str term_str  \
chap_num para_num sent_num token_num                                         
1        0        0        0          (Alive, NNP)  NNP     Alive    alive   
                           1           (again, RB)   RB     again    again   
                  1        0            (Then, RB)   RB      Then     then   
                           1            (show, VB)   VB      show     show   
                           2             (me, PRP)  PRP        me       me   

                                     pos_group  
chap_num para_num sent_num token_num            
1        0        0        0                NN  
                           1                RB  
                  1        0                RB  
                           1                VB  
                           2                PR

In [79]:
# token_pat = r"[\s',-]+"
# TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
#     .to_frame('token_str')
# TOKENS.index.names = OHCO[:4]
# TOKENS
# TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
# VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
# VOCAB.index.name = 'term_id'
# VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [80]:
TOKENS.head()

pos_tuple  pos token_str term_str  \
chap_num para_num sent_num token_num                                         
1        0        0        0          (Alive, NNP)  NNP     Alive    alive   
                           1           (again, RB)   RB     again    again   
                  1        0            (Then, RB)   RB      Then     then   
                           1            (show, VB)   VB      show     show   
                           2             (me, PRP)  PRP        me       me   

                                     pos_group  
chap_num para_num sent_num token_num            
1        0        0        0                NN  
                           1                RB  
                  1        0                RB  
                           1                VB  
                           2                PR

In [81]:
#VOCAB.head()

NameError: name 'VOCAB' is not defined

In [82]:
melmoth_CHAPS = CHAPS
melmoth_PARAS = PARAS
melmoth_SENTS = SENTS
melmoth_TOKENS = TOKENS
#melmoth_VOCAB = VOCAB

In [83]:
print("Melmoth len: ", len(melmoth_TOKENS))

Melmoth len:  66304


In [84]:
melmoth_TOKENS.to_csv(f"{output_dir}/melmoth-TOKENS.csv")


In [85]:
high_goth_corp = [udolpho_TOKENS, baron_TOKENS, otranto_TOKENS]
ids = [3268, 5182, 696]
HG_CORPUS = pd.concat(high_goth_corp, keys=ids, names=['doc_id'])
HG_CORPUS.to_csv(f"{output_dir}/hg-corpus.csv")

In [86]:
HG_CORPUS

pos_tuple   pos  \
doc_id chap_num para_num sent_num token_num                           
3268   1        0        0        0                (home, NN)    NN   
                                  1                 (is, VBZ)   VBZ   
                                  2                 (the, DT)    DT   
                                  3              (resort, NN)    NN   
                                  4                  (Of, IN)    IN   
...                                                       ...   ...   
696    5        121      5        41             (taken, VBN)   VBN   
                                  42         (possession, NN)    NN   
                                  43                 (of, IN)    IN   
                                  44              (his, PRP$)  PRP$   
                                  45               (soul, NN)    NN   

                                              token_str    term_str pos_group  
doc_id chap_num para_num sent_num token_num                                    
3268   1        0        0        0                home        home        NN  
                                  1                  is          is        VB  
                                  2                 the         the        DT  
                                  3              resort      resort        NN  
                                  4                  Of          of        IN  
...                                                 ...         ...       ...  
696    5        121      5        41              taken       taken        VB  
                                  42         possession  possession        NN  
                                  43                 of          of        IN  
                                  44                his         his        PR  
                                  45               soul        soul        NN  

[184412 rows x 5 columns]